<a href="https://colab.research.google.com/github/irAbs174/openshell-notebook/blob/main/openshell_notobook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Открыть в Colab"/></a>

# Современная изоляция AI-агентов с NVIDIA OpenShell: Полная архитектура и пошаговый практический блокнот

**NVIDIA OpenShell** — это открытая среда выполнения с политиками, разработанная специально для автономных, саморазвивающихся AI-агентов. В отличие от традиционных сред выполнения контейнеров, которые изолируют типовые рабочие нагрузки, OpenShell предоставляет **внепроцессные, бездоверительные (zero-trust) экологические ограничения**.

AI-агент, работающий внутри песочницы OpenShell, может саморазвиваться, писать новые Python-скрипты на лету и устанавливать инструменты — но при этом он полностью неспособен похищать учетные данные, обходить разрешенные пути файловой системы или совершать несанкционированные исходящие сетевые запросы.

---

## 1. Углубленная архитектура и основные концепции

OpenShell использует модель безопасности, аналогичную песочнице вкладок современного веб-браузера:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                           ХОСТ-ОКРУЖЕНИЕ                                    │
│                                                                             │
│  ┌───────────────────────┐             ┌─────────────────────────────────┐  │
│  │     CLI OpenShell     │             │        openshell-gateway        │  │
│  │  (`openshell sandbox`)│             │   (gRPC / HTTP Сервер :17670)   │  │
│  └───────────┬───────────┘             └────────────────┬────────────────┘  │
│              │                                          │                   │
│              └──────────────────┐  ┌────────────────────┘                   │
│                                 ▼  ▼                                        │
│  ┌───────────────────────────────────────────────────────────────────────┐  │
│  │                    СРЕДА ИСПОЛНЕНИЯ ПЕСОЧНИЦЫ                        │  │
│  │  ┌─────────────────────────────────────────────────────────────────┐  │  │
│  │  │                        Обвязка Агента                          │  │  │
│  │  │                 (Claude Code, Codex, Пользовательская)         │  │  │
│  │  └──────────────────────────────┬──────────────────────────────────┘  │  │
│  │                                 │ (Выполняет команды)                 │  │
│  │                                 ▼                                     │  │
│  │  ┌─────────────────────────────────────────────────────────────────┐  │  │
│  │  │                   ВНЕПРОЦЕССНАЯ ПОЛИТИКА                       │  │  │
│  │  ├─────────────────────────────────────────────────────────────────┤  │  │
│  │  │ 1. Файловая система   : Песочница Landlock Linux               │  │  │
│  │  │ 2. Сетевой уровень    : Фильтрация L7 Прокси (Хост, API, Исх.) │  │  │
│  │  │ 3. Маршрутизатор      : Бездоверительная замена учетных данных │  │  │
│  │  └─────────────────────────────────────────────────────────────────┘  │  │
│  └───────────────────────────────────────────────────────────────────────┘  │  │
└─────────────────────────────────────────────────────────────────────────────┘

```

### Ключевые архитектурные столпы

1. **Внепроцессное применение политик**: Средства безопасности находятся *вне* контекстного окна агента и границ процесса. Даже если злоумышленник успешно внедрит инъекцию в запрос агента, ядро и шлюз OpenShell откажут в запрещенных системных вызовах и сетевых маршрутах.
2. **Изоляция файловой системы Linux Landlock**: Применяет детализированные правила чтения/записи на уровне каталогов на уровне ядра Linux.
3. **Прокси L7 и сетевая фильтрация**: Исходящие HTTP/HTTPS-запросы проходят через прозрачный прокси с политиками. Несанкционированные конечные точки немедленно блокируются с кодом состояния `403`.
4. **Маршрутизатор конфиденциальности (изоляция учетных данных)**: Хранит ключи API провайдеров LLM за пределами песочницы. Агент направляет локальные запросы на внутренний маршрут, а OpenShell подставляет авторизованные бэкенд-учетные данные на лету.

---

## 2. Высокопроизводительный интерактивный Jupyter-блокнот

Скопируйте и вставьте ячейки кода ниже непосредственно в `.ipynb`-блокнот. Последовательное выполнение ячеек проведет вас от нуля до запуска изолированного агента с пользовательскими декларативными политиками.

---

### Markdown-ячейка 1: Настройка окружения и диагностика

```markdown
# Раздел 1: Диагностика системы OpenShell и проверка окружения
В этом разделе мы проверяем хост-окружение, наличие доступных драйверов выполнения (Docker, Podman или Kubernetes) и просматриваем бинарные файлы OpenShell.

```

### Ячейка кода 1

In [ ]:
import os
import shutil
import subprocess
import sys


def run_command(cmd, verbose=True):
  """Выполняет команды оболочки и безопасно выводит потоковые данные внутри Jupyter."""
  print(f"\033[1;34m[ВЫПОЛНЕНИЕ]\033[0m {cmd}")
  process = subprocess.Popen(
      cmd,
      shell=True,
      stdout=subprocess.PIPE,
      stderr=subprocess.PIPE,
      text=True,
  )

  stdout_lines, stderr_lines = [], []
  while True:
    output = process.stdout.readline()
    if output == "" and process.poll() is not None:
      break
    if output and verbose:
      print(output.strip())
      stdout_lines.append(output)

  _, stderr = process.communicate()
  if stderr and verbose:
    print(f"\033[1;31m[СТДЕРР]\033[0m\n{stderr.strip()}")

  return process.returncode, "".join(stdout_lines), stderr


# 1. Проверка системных инструментов
tools = ["openshell", "openshell-gateway", "docker", "curl"]
for tool in tools:
  path = shutil.which(tool)
  status = f"\033[1;32mНАЙДЕН\033[0m по адресу {path}" if path else "\033[1;31mНЕ НАЙДЕН\033[0m"
  print(f"Проверка инструмента [{tool}]: {status}")

# 2. Проверка версий OpenShell
run_command("openshell --version")
run_command("openshell-gateway --version")

---

### Markdown-ячейка 2: Запуск и настройка `openshell-gateway`

```markdown
# Раздел 2: Запуск сервиса OpenShell Gateway
`openshell-gateway` — это фоновый демон gRPC/HTTP, управляющий жизненным циклом песочниц, mTLS-сертификатами клиентов и оценкой политик.

```

### Ячейка кода 2

In [ ]:
import time

# Определение конфигурации окружения для локального шлюза разработки
gateway_port = 17670
gateway_env = os.environ.copy()
gateway_env["OPENSHELL_BIND_ADDRESS"] = "127.0.0.1"
gateway_env["OPENSHELL_SERVER_PORT"] = str(gateway_port)
gateway_env["OPENSHELL_LOG_LEVEL"] = "info"
gateway_env["OPENSHELL_DISABLE_TLS"] = "true"  # Режим локальной разработки

print("Запуск openshell-gateway в фоновом режиме...")
gateway_proc = subprocess.Popen(
    [
        "openshell-gateway",
        "--bind-address",
        "127.0.0.1",
        "--port",
        str(gateway_port),
        "--disable-tls",
        "--log-level",
        "info",
    ],
    env=gateway_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

# Ожидание инициализации шлюза
time.sleep(3)

# Проверка подключения к шлюзу через CLI doctor/status
code, out, err = run_command(
    f"openshell --gateway-endpoint http://127.0.0.1:{gateway_port} gateway"
    " doctor"
)

# Регистрация локального шлюза в конфигурации метаданных
run_command(
    f"openshell gateway add http://127.0.0.1:{gateway_port} --local"
    " --name local-dev"
)

---

### Markdown-ячейка 3: Создание и проверка песочницы

```markdown
# Раздел 3: Управление жизненным циклом песочницы
Мы создадим изолированную среду выполнения и изучим ограничения безопасности по умолчанию.

```

### Ячейка кода 3

In [ ]:
# 1. Создание изолированной песочницы с именем 'agent-demo'
run_command("openshell sandbox create --name agent-demo")

# 2. Список активных песочниц
run_command("openshell sandbox list")

# 3. Проверка исходящей сетевой доступности внутри песочницы по умолчанию
# Исходящие соединения должны отклоняться движком политик L7 по умолчанию.
test_net_cmd = (
    "openshell exec agent-demo -- curl -sS --connect-timeout 5"
    " https://api.github.com/zen"
)
run_command(test_net_cmd)

---

### Markdown-ячейка 4: Создание и применение декларативных политик

```markdown
# Раздел 4: Динамическое внедрение политик
Политики OpenShell управляют файловой системой, сетевыми маршрутами L7 и выполнением процессов без необходимости перезапуска песочницы.

```

### Ячейка кода 4

In [ ]:
# Создание строгого YAML-файла политики
policy_content = """
apiVersion: v1alpha1
kind: SandboxPolicy
metadata:
  name: demo-github-read-only
spec:
  network:
    egress:
      - match:
          host: "api.github.com"
          scheme: "https"
        rules:
          - methods: ["GET"]
            path: "/*"
            action: Allow
          - methods: ["POST", "PUT", "DELETE"]
            path: "/*"
            action: Deny
  filesystem:
    readOnlyPaths:
      - "/usr"
      - "/lib"
    readWritePaths:
      - "/tmp"
      - "/workspace"
  process:
    allowExecution:
      - "/bin/*"
      - "/usr/bin/*"
"""

policy_filename = "github_policy.yaml"
with open(policy_filename, "w") as f:
  f.write(policy_content.strip())

print(f"Создан файл политики: {policy_filename}")

# Динамическое применение политики к запущенной песочнице
run_command(f"openshell policy set agent-demo --policy {policy_filename} --wait")

---

### Markdown-ячейка 5: Проверка средств безопасности (тесты исходящего трафика и методов)

```markdown
# Раздел 5: Проверка применения политик
Мы тестируем HTTP GET (Разрешено) против HTTP POST (Запрещено) на `api.github.com`.

```

### Ячейка кода 5

In [ ]:
print("--- ТЕСТ 1: HTTP GET (Должен выполниться успешно) ---")
run_command(
    "openshell exec agent-demo -- curl -sS -X GET https://api.github.com/zen"
)

print("\n--- ТЕСТ 2: HTTP POST (Должен быть ЗАБЛОКИРОВАН прокси L7) ---")
run_command(
    "openshell exec agent-demo -- curl -sS -X POST"
    " https://api.github.com/user/repos -d '{\"name\":\"exploit\"}'"
)

print("\n--- ТЕСТ 3: Неразрешенный исходящий хост (Должен быть ЗАБЛОКИРОВАН) ---")
run_command(
    "openshell exec agent-demo -- curl -sS --connect-timeout 3"
    " https://example.com"
)

---

### Markdown-ячейка 6: Просмотр журналов аудита песочницы и очистка

```markdown
# Раздел 6: Журналирование аудита безопасности и очистка песочницы
OpenShell регистрирует каждый перехваченный запрос, нарушение политики и событие песочницы.

```

### Ячейка кода 6

In [ ]:
# Получение журналов безопасности
print("=== ЖУРНАЛЫ АУДИТА ПЕСОЧНИЦЫ ===")
run_command("openshell logs agent-demo --tail 20")

# Очистка ресурсов песочницы
print("\n=== ОЧИСТКА ===")
run_command("openshell sandbox delete agent-demo --force")

# Остановка процесса шлюза
if 'gateway_proc' in locals():
  gateway_proc.terminate()
  print("Сервер OpenShell Gateway успешно остановлен.")

---

## 3. Краткий справочник команд CLI и шлюза

| Подкоманда OpenShell | Описание |
| --- | --- |
| `openshell sandbox create` | Создает новую изолированную песочницу выполнения |
| `openshell policy set <имя> -p <файл>` | Применяет политики YAML с горячей перезагрузкой к песочнице |
| `openshell logs <имя>` | Выводит в реальном времени журналы аудита сети и файловой системы |
| `openshell-gateway --config <файл>` | Запускает центральный демон-шлюз выполнения |
| `openshell-gateway generate-certs` | Генерирует PKI-сертификаты для mTLS-аутентификации |

---

## 4. Продвинутые паттерны для production и корпоративные функции

Для полноценной интеграции OpenShell через Jupyter вот три важных паттерна для запуска production-нагрузок агентов.

---

### Паттерн A: Мультитенантная аутентификация OIDC и JWT

При развертывании `openshell-gateway` в многопользовательском кластере или корпоративной платформе аутентификация переходит от локальных mTLS-сертификатов к валидации JWT через OpenID Connect (OIDC).

#### Конфигурация запуска шлюза

```bash
openshell-gateway \
  --bind-address 0.0.0.0 \
  --port 17670 \
  --oidc-issuer https://auth.yourcompany.com/realms/agents \
  --oidc-audience openshell-cli \
  --oidc-roles-claim realm_access.roles \
  --oidc-admin-role openshell-admin \
  --oidc-user-role openshell-user \
  --drivers kubernetes

```

#### Выполнение аутентифицированных команд в Python

In [ ]:
import os
import subprocess

# Получение JWT-токена доступа от провайдера идентификации (например, Keycloak, Entra ID, Okta)
jwt_token = os.environ.get("OPENID_ACCESS_TOKEN")

# Целевая конечная точка шлюза с передачей bearer-токена
cmd = [
    "openshell",
    "--gateway-endpoint",
    "https://openshell-gateway.internal.net:17670",
    "sandbox",
    "create",
    "--name",
    "authenticated-user-agent",
]

env = os.environ.copy()
env["OPENSHELL_BEARER_TOKEN"] = jwt_token

result = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(result.stdout)

---

### Паттерн B: Изолированный многоступенчатый конвейер агента (выполнение и проверка кода)

Типичный рабочий процесс саморазвивающихся агентов включает разделение **генерации кода** и **выполнения ненадежного кода**. Вы можете организовать две песочницы с разными политиками внутри Jupyter:

```
┌───────────────────────────┐         ┌───────────────────────────┐
│   Песочница Генератора    │         │   Песочница Исполнителя   │
│  • Сеть: Исход. к LLM    │  ────►  │  • Сеть: Изолирована      │
│  • ФС: Только запись     │  Код    │  • ФС: Только чтение      │
└───────────────────────────┘         └───────────────────────────┘

```

In [ ]:
import subprocess


class DualStageAgentWorkflow:

  def __init__(self, gateway_url="http://127.0.0.1:17670"):
    self.gateway = gateway_url

  def run_pipeline(self, python_code_to_verify: str):
    # 1. Создание ненадежной песочницы-исполнителя
    subprocess.run(
        f"openshell --gateway-endpoint {self.gateway} sandbox create --name"
        " evaluator",
        shell=True,
        check=True,
    )

    # 2. Создание ограничительной автономной политики (нулевой исходящий трафик)
    offline_policy = """
apiVersion: v1alpha1
kind: SandboxPolicy
metadata:
  name: total-isolation
spec:
  network:
    egress: []  # Исходящие соединения запрещены
  filesystem:
    readOnlyPaths: ["/usr", "/lib"]
    readWritePaths: ["/tmp"]
"""
    with open("/tmp/offline_policy.yaml", "w") as f:
      f.write(offline_policy)

    subprocess.run(
        f"openshell --gateway-endpoint {self.gateway} policy set evaluator"
        " --policy /tmp/offline_policy.yaml --wait",
        shell=True,
        check=True,
    )

    # 3. Безопасное выполнение ненадежного кода в оценщике с нулевым доверием
    exec_cmd = (
        f"openshell --gateway-endpoint {self.gateway} exec evaluator -- python3"
        f" -c '{python_code_to_verify}'"
    )
    res = subprocess.run(exec_cmd, shell=True, capture_output=True, text=True)

    print("=== ВЫВОД ЭТАПА ОЦЕНКИ ===")
    print("СТДИВ:", res.stdout)
    print("СТДЕРР:", res.stderr)
    print("Код возврата:", res.returncode)

    # 4. Очистка
    subprocess.run(
        f"openshell --gateway-endpoint {self.gateway} sandbox delete evaluator"
        " --force",
        shell=True,
    )


# Запуск теста конвейера
pipeline = DualStageAgentWorkflow()
pipeline.run_pipeline("import sys; print('Безопасная оценка ненадежного кода!')")

---

### Паттерн C: Интеграция наблюдаемости OpenShell с метриками Prometheus

При запуске `openshell-gateway` с параметром `--metrics-port 9090` он открывает конечную точку с метриками Prometheus для мониторинга песочниц, нарушений исходящего трафика и производительности системы в реальном времени.

In [ ]:
import urllib.request

# Запрос к конечной точке метрик Prometheus OpenShell
metrics_url = "http://127.0.0.1:9090/metrics"

try:
  with urllib.request.urlopen(metrics_url) as response:
    metrics_data = response.read().decode("utf-8")

  # Фильтрация метрик для нарушений исходящего трафика и активных песочниц
  relevant_metrics = [
      line
      for line in metrics_data.split("\n")
      if "openshell_sandbox" in line or "openshell_policy_violations" in line
  ]

  print("=== МЕТРИКИ ВЫПОЛНЕНИЯ OPENSHELL ===")
  for metric in relevant_metrics[:10]:
    print(metric)

except Exception as e:
  print(f"Конечная точка метрик неактивна или недоступна: {e}")